# Cell GLM model



In [268]:
# Imports
import sys
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import holoviews as hv
import hvplot.pandas
hv.extension('bokeh')

from pprint import pprint

# Add parent directory to path
sys.path.insert(0, str(Path.cwd().parent))

# Import the class
import importlib
import session_class
importlib.reload(session_class)
from session_class import Session
from matplotlib import pyplot as plt

# Also reload multi_session_pca if needed
import multi_session_pca
importlib.reload(multi_session_pca)
from multi_session_pca import MultiSessionPCA

print("✓ Imports loaded successfully!")

✓ Imports loaded successfully!


## 1. Load cell via session

In [269]:
# Load MSN cell database
monkey = 'fiona'  # 'yasmin' or 'fiona'

base_path = Path.cwd().parent / 'data' / 'unified_cell_trial_data' 
pickle_file = base_path / f'msn_{monkey}_cell_trial_data.pkl'

cell_df = pd.read_pickle(pickle_file)

In [270]:
# Add parent directory to path to import Cell and PopulationAnalyzer
sys.path.insert(0, str(Path.cwd().parent))
import importlib
import session_class
import cell_analysis
import pca_helpers
importlib.reload(session_class)
importlib.reload(cell_analysis)
importlib.reload(pca_helpers)
from session_class import Session
from pca_helpers import extract_session_psth_worker
from cell_analysis import Cell, PopulationAnalyzer

print("Cell, PopulationAnalyzer, and Session classes imported successfully!")

Cell, PopulationAnalyzer, and Session classes imported successfully!


In [271]:
# Select a session to analyze - use session with most cells
session_cell_counts = cell_df.groupby('trial_session')['cell_ID'].nunique().sort_values(ascending=False)
best_session_id = session_cell_counts.index[0]

print(f"Using session with most cells: {best_session_id}")
print(f"Number of cells in this session: {session_cell_counts.iloc[0]}")

# Get all data for this session
session_data = cell_df[cell_df['trial_session'] == best_session_id]

# Create Session object
session = Session(session_data, verbose=True)


Using session with most cells: fi211110a
Number of cells in this session: 85
Session fi211110a initialized:
  - Number of cells: 85
  - Total trials: 943
  - Trial types: ['CONT', 'GO', 'STOP']
  - Directions: [np.int64(0), np.int64(180)]


In [272]:
cell = session.get_cell(session.cell_ids[i])
print(f"Analyzing Cell ID: {cell.cell_id}:\n{cell}")

Analyzing Cell ID: 2105:
Cell 2105 | Type: msn | Trials: 921 | Directions: [np.int64(0), np.int64(180)] | Trial Types: ['CONT', 'GO', 'STOP'] | SSDs: [np.float64(1.0), np.float64(2.0), np.float64(3.0), np.float64(4.0)]


## 2. Anove classification

In [273]:
from scipy import stats

def perform_cell_anova(cell):
    """
    Perform ANOVA tests to determine cell sensitivity to various conditions.
    
    Conditions tested:
    1. Task Modulation: Baseline vs. GO Phase (All directions)
    2. GO Directionality: GO Left vs. GO Right (during GO phase)
    3. STOP Directionality: STOP Left vs. STOP Right (during STOP phase)
    4. CONT Directionality: CONT Left vs. CONT Right (during CONT phase)
    5. Signal Sensitivity: STOP vs. CONT (aligned to STOP cue)
    """
    
    results = {}
    
    # Helper to extract firing rates manually
    def get_rates(align_event, window, trial_type=None, direction=None, success_only=True):
        # Filter trials
        df = cell.filter_trials(trial_type=trial_type, direction=direction, success_only=success_only)
        
        rates = []
        for _, row in df.iterrows():
            # Get alignment time
            t0 = np.nan
            
            if align_event == 'go_cue':
                t0 = row['go_cue']
            elif align_event == 'stop_cue':
                # Both STOP and CONT trials have a stop_cue
                t0 = row['stop_cue']
            
            # Skip if alignment event is missing
            if pd.isna(t0):
                continue
                
            spikes = np.array(row['neural_data'])
            # Align spikes
            aligned_spikes = spikes - t0
            
            # Count spikes in window
            count = np.sum((aligned_spikes >= window[0]) & (aligned_spikes <= window[1]))
            duration = (window[1] - window[0]) / 1000.0
            rates.append(count / duration)
            
        return np.array(rates)

    # 1. Task Modulation (Baseline vs GO)
    # Baseline: [-500, 0] aligned to GO
    # GO: [0, 500] aligned to GO
    base_rates = get_rates('go_cue', [-500, 0], trial_type='GO')
    go_rates = get_rates('go_cue', [0, 500], trial_type='GO')
    
    if len(base_rates) > 0 and len(go_rates) > 0:
        f_stat, p_val = stats.f_oneway(base_rates, go_rates)
        results['Task_Modulation'] = {'F': f_stat, 'p': p_val, 'significant': p_val < 0.05}
    else:
        results['Task_Modulation'] = {'F': np.nan, 'p': np.nan, 'significant': False}
    
    # 2. GO Directionality
    # GO Left (180) vs GO Right (0) in [0, 500] aligned to GO
    go_left = get_rates('go_cue', [0, 500], trial_type='GO', direction=180)
    go_right = get_rates('go_cue', [0, 500], trial_type='GO', direction=0)
    
    if len(go_left) > 0 and len(go_right) > 0:
        f_stat, p_val = stats.f_oneway(go_left, go_right)
        results['GO_Directionality'] = {'F': f_stat, 'p': p_val, 'significant': p_val < 0.05}
    else:
        results['GO_Directionality'] = {'F': np.nan, 'p': np.nan, 'significant': False}

    # 3. STOP Directionality
    # STOP Left vs STOP Right in [0, 200] aligned to STOP cue
    stop_left = get_rates('stop_cue', [0, 200], trial_type='STOP', direction=180)
    stop_right = get_rates('stop_cue', [0, 200], trial_type='STOP', direction=0)
    
    if len(stop_left) > 0 and len(stop_right) > 0:
        f_stat, p_val = stats.f_oneway(stop_left, stop_right)
        results['STOP_Directionality'] = {'F': f_stat, 'p': p_val, 'significant': p_val < 0.05}
    else:
        results['STOP_Directionality'] = {'F': np.nan, 'p': np.nan, 'significant': False}

    # 4. CONT Directionality
    # CONT Left vs CONT Right in [0, 500] aligned to GO (since no stop cue)
    # Using same window as GO
    cont_left = get_rates('go_cue', [0, 500], trial_type='CONT', direction=180)
    cont_right = get_rates('go_cue', [0, 500], trial_type='CONT', direction=0)
    
    if len(cont_left) > 0 and len(cont_right) > 0:
        f_stat, p_val = stats.f_oneway(cont_left, cont_right)
        results['CONT_Directionality'] = {'F': f_stat, 'p': p_val, 'significant': p_val < 0.05}
    else:
        results['CONT_Directionality'] = {'F': np.nan, 'p': np.nan, 'significant': False}

    # 5. Signal Sensitivity (STOP vs CONT)
    # Compare STOP vs CONT aligned to STOP cue
    # Window: [0, 200] ms after STOP cue
    stop_rates_sig = get_rates('stop_cue', [0, 200], trial_type='STOP')
    cont_rates_sig = get_rates('stop_cue', [0, 200], trial_type='CONT')
    
    if len(stop_rates_sig) > 0 and len(cont_rates_sig) > 0:
        f_stat, p_val = stats.f_oneway(stop_rates_sig, cont_rates_sig)
        results['Signal_Sensitivity'] = {'F': f_stat, 'p': p_val, 'significant': p_val < 0.05}
    else:
        results['Signal_Sensitivity'] = {'F': np.nan, 'p': np.nan, 'significant': False}

    return results

# Run analysis
anova_results = perform_cell_anova(cell)
print(f"ANOVA Results for Cell {cell.cell_id}:")
pprint(anova_results)

ANOVA Results for Cell 2105:
{'CONT_Directionality': {'F': np.float64(5.8854023438666765),
                         'p': np.float64(0.016186612832561786),
                         'significant': np.True_},
 'GO_Directionality': {'F': np.float64(25.487861783979916),
                       'p': np.float64(6.27948009640434e-07),
                       'significant': np.True_},
 'STOP_Directionality': {'F': np.float64(2.160009118574256),
                         'p': np.float64(0.14449823585325106),
                         'significant': np.False_},
 'Signal_Sensitivity': {'F': np.float64(2.5993954077693204),
                        'p': np.float64(0.10793872204418983),
                        'significant': np.False_},
 'Task_Modulation': {'F': np.float64(76.86286283188167),
                     'p': np.float64(7.904023722430811e-18),
                     'significant': np.True_}}


In [274]:
# Check if CONT trials have stop_cue values
cont_trials = cell.data[cell.data['type'] == 'CONT']
print(f"Number of CONT trials: {len(cont_trials)}")
print(f"Number of CONT trials with valid stop_cue: {cont_trials['stop_cue'].notna().sum()}")
print("Sample CONT trials:")
print(cont_trials[['type', 'go_cue', 'stop_cue', 'ssd_number']].head())

Number of CONT trials: 219
Number of CONT trials with valid stop_cue: 219
Sample CONT trials:
   type  go_cue  stop_cue  ssd_number
0  CONT     984    1032.0         1.0
1  CONT    1049    1097.0         1.0
2  CONT     949     997.0         1.0
3  CONT     954    1002.0         1.0
4  CONT    1005    1053.0         1.0


In [280]:
i = 16
epok = [-500, 500]
align = "go_cue"
type = 'GO'

In [281]:
cell = session.get_cell(session.cell_ids[i])
print(f"i = {i}")
i += 1
rasters = cell.plot_raster_by_type_direction(
    epok=epok, alignment_point=align, 
)
rasters
(rasters[0][type] + rasters[180][type]).cols(1)

i = 16


:Layout
   .Overlay.I  :Overlay
      .Curve.I   :Curve   [x]   (y)
      .HeatMap.I :HeatMap   [columns,index]   (value)
   .Overlay.II :Overlay
      .Curve.I   :Curve   [x]   (y)
      .HeatMap.I :HeatMap   [columns,index]   (value)

In [282]:
psths = cell.plot_psth_by_type_direction(
    epok = epok,
    bin_size = 1,
    alignment_point = align,
    separate_ssd = False,
    smooth = True,
    smooth_ker_size = 25,
    delta = False,
    normalize_bins = False
)

psths[0][type] * psths[180][type]

:Overlay
   .Curve.I  :Curve   [Time]   (Firing Rate (spikes/s))
   .VLine.I  :VLine   [x,y]
   .Curve.II :Curve   [Time]   (Firing Rate (spikes/s))
   .VLine.II :VLine   [x,y]

In [283]:
psths[0]["CONT"] * psths[180]["CONT"]

:Overlay
   .Curve.I  :Curve   [Time]   (Firing Rate (spikes/s))
   .VLine.I  :VLine   [x,y]
   .Curve.II :Curve   [Time]   (Firing Rate (spikes/s))
   .VLine.II :VLine   [x,y]

In [284]:
trials = np.sort(cell.data['trial_number'].value_counts().index.to_numpy())
trials.shape, trials[-1] - trials[0]

((437,), np.int64(436))